# Spending Analyzer: A2 experiments

This notebook reruns every number in our A2 slides. It only uses pandas, numpy and scikit-learn, so it works on a fresh Google Colab runtime.

**How to run:** put `transactions.csv` in the same folder as this notebook (or upload it when the first cell asks), then do Runtime → Run all. It takes about a minute.

**The data:** 211 made-up charges for one student, July to September 2026, written in the same formats real bank and card statements use. We labeled each charge with one of 11 categories by hand. For stores that sell many kinds of things (Target, Costco, Stewart's...), the label is what was actually bought, which is in the `note` column.

Sections:
1. Clean up the bank text
2. Keyword rules
3. Trained models on stores they've seen vs. new stores
4. Best possible score from the text alone
5. Classifier confidence on new stores (the failure on slide 5)
6. Similarity scores for look-alike charges
7. Recurring-bill check
8. September totals (the answer key for the LLM test)
9. Scoring the LLM replies (optional, needs the `llm` folder)
10. The numbers on the slides

In [ ]:
import html
import json
import os
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

SEED = 4150
DATA = "transactions.csv"
if not os.path.exists(DATA):
    try:
        from google.colab import files  # only exists on Colab
        print("Please upload transactions.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put transactions.csv in the same folder as this notebook.")

df = pd.read_csv(DATA)
y = df["category"].to_numpy()
groups = df["merchant"].to_numpy()
print(len(df), "charges,", df["merchant"].nunique(), "merchants,", df["category"].nunique(), "categories")
df.head(8)

## 1. Clean up the bank text

Banks wrap the store name in extra codes: `POS PURCHASE`, `CHECKCARD 0904`, card numbers, store numbers. This step strips them so that the same store looks the same every time.

In [ ]:
PREFIXES = [r"^POS PURCHASE ", r"^CHECKCARD \d{4} ", r"^PURCHASE AUTHORIZED ON \d\d/\d\d "]


def normalize(desc: str) -> str:
    s = desc.upper()
    for p in PREFIXES:
        s = re.sub(p, "", s)
    s = re.sub(r" CARD \d{4}$", "", s)
    s = re.sub(r"(PPD|WEB) ID: ?\d+", "", s)
    s = re.sub(r"REF #\S+", "", s)
    s = s.replace("*", " * ")
    tokens = [t for t in s.split() if not re.search(r"\d", t)]
    return re.sub(r"\s+", " ", " ".join(tokens)).strip()


df["clean"] = df["description"].map(normalize)
df[["description", "clean"]].sample(8, random_state=1)

## 2. Keyword rules

We wrote these before looking at any results. The first keyword that matches decides the category. Notice the generic `FEE` rule: it also catches the gym (`PLANET FIT CLUB FEES`).

In [ ]:
RULES = [
    ("Income", ["PAYROLL", "DIRECT DEP", "DIR DEP"]),
    ("Transfers", ["VENMO", "ZELLE", "TRANSFER", "MOBILE PMT", "PAYMENT THANK YOU", "CASHOUT"]),
    ("Fees", ["SERVICE FEE", "OVERDRAFT", "ATM FEE", "LATE FEE", "LATE PAYMENT", "VIOLATION", "FEE"]),
    ("Rent & Utilities", ["RENT", "NATIONAL GRID", "SPECTRUM", "VERIZON", "T-MOBILE", "MINT MOBILE", "ELECTRIC",
                          "UTILIT", "COMCAST", "XFINITY"]),
    ("Subscriptions", ["NETFLIX", "SPOTIFY", "HULU", "DISNEY", "APPLE.COM/BILL", "CHATGPT", "OPENAI",
                       "AMAZON PRIME", "PATREON", "YOUTUBE"]),
    ("Dining", ["STARBUCKS", "DUNKIN", "CHIPOTLE", "MCDONALD", "DOORDASH", "GRUBHUB", "UBER * EATS", "UBER EATS",
                "PANERA", "CAFE", "COFFEE", "PIZZA", "BAGEL", "BBQ", "BAR-B-Q", "GRILL", "RESTAURANT"]),
    ("Groceries", ["PRICE CHOPPER", "HANNAFORD", "ALDI", "TRADER JOE", "WHOLE FOODS", "WHOLEFDS", "WEGMANS",
                   "INSTACART", "COSTCO", "GROCERY", "MARKET", "SUPERMARKET", "FOOD"]),
    ("Transport", ["UBER", "LYFT", "CDTA", "SHELL", "MOBIL", "SUNOCO", "EXXON", "STEWARTS", "GAS", "FUEL",
                   "PARKING", "AMTRAK", "MTA", "TRANSIT"]),
    ("Shopping", ["AMAZON", "AMZN", "TARGET", "WALMART", "WAL-MART", "WM SUPERCENTER", "BEST BUY", "ETSY", "EBAY",
                  "NIKE", "UNIQLO", "BOOKSTORE", "SHEIN"]),
    ("Health", ["CVS", "WALGREENS", "PHARMACY", "FITNESS", "URGENT CARE", "MEDICAL", "DENTAL", "COPAY"]),
    ("Entertainment", ["AMC", "REGAL", "STEAM", "TICKETMASTER", "PLAYSTATION", "XBOX", "NINTENDO", "BOWL",
                       "CINEMA", "THEATRE", "THEATER"]),
]
print(sum(len(k) for _, k in RULES), "keywords")


def rule_label(clean: str):
    for cat, kws in RULES:
        for kw in kws:
            if kw in clean:
                return cat, kw
    return None, None


rl = df["clean"].map(rule_label)
df["rule_pred"] = [r[0] for r in rl]
df["rule_kw"] = [r[1] for r in rl]
fired = df["rule_pred"].notna()
print(f"A rule fired on {fired.mean():.0%} of charges")
print(f"Right when a rule fired: {(df.loc[fired, 'rule_pred'] == df.loc[fired, 'category']).mean():.0%}")
print(f"Accuracy on all charges: {(df['rule_pred'] == df['category']).mean():.0%}")
df.loc[fired & (df["rule_pred"] != df["category"]), ["clean", "category", "rule_pred", "rule_kw"]].drop_duplicates("clean")

## 3. Trained models on stores they've seen vs. new stores

Three models learn from our hand labels:
- **Classifier:** TF-IDF on character pieces of the text (3 to 5 letters) + logistic regression.
- **Neural net:** a small MLP (one hidden layer of 64 units) on the same features.
- **Similarity search:** find the most similar charge we already labeled (cosine similarity) and copy its category.

Two ways to test, both with 5-fold cross-validation:
- **Seen stores:** random folds, so the same store usually shows up in training.
- **New stores:** every charge from a store is held out together (`StratifiedGroupKFold` grouped by merchant), so the model never saw that store in training.

We also try whole words instead of character pieces, and raw vs. cleaned text.

In [ ]:
seen_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
unseen_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)


def char_vec():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)


def word_vec():
    return TfidfVectorizer(analyzer="word", token_pattern=r"[A-Za-z][A-Za-z'.&/-]+", ngram_range=(1, 2),
                           sublinear_tf=True)


def lr():
    return LogisticRegression(max_iter=5000, C=10)


def mlp():
    return MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-4, max_iter=3000, random_state=SEED)


def cv_predict(make_model, X, cv, use_groups):
    pred = np.empty(len(X), dtype=object)
    proba_max = np.zeros(len(X))
    for tr, te in cv.split(X, y, groups if use_groups else None):
        m = make_model()
        m.fit(X[tr], y[tr])
        pred[te] = m.predict(X[te])
        proba_max[te] = m.predict_proba(X[te]).max(axis=1)
    return pred, proba_max


def nn_predict(X, cv, use_groups):
    """Similarity search: copy the category of the most similar labeled charge."""
    pred = np.empty(len(X), dtype=object)
    sim = np.zeros(len(X))
    nbr = np.empty(len(X), dtype=object)
    for tr, te in cv.split(X, y, groups if use_groups else None):
        v = char_vec().fit(X[tr])
        A, B = v.transform(X[tr]), v.transform(X[te])
        S = (B @ A.T).toarray()  # rows are L2-normalised, so this is cosine similarity
        best = S.argmax(axis=1)
        pred[te] = y[tr][best]
        sim[te] = S[np.arange(len(te)), best]
        nbr[te] = X[tr][best]
    return pred, sim, nbr


X_raw = df["description"].to_numpy()
X_clean = df["clean"].to_numpy()

runs = {}
for split_name, cv, g in [("seen", seen_cv, False), ("unseen", unseen_cv, True)]:
    for feat_name, X in [("raw", X_raw), ("clean", X_clean)]:
        runs[f"lr_char_{feat_name}_{split_name}"] = cv_predict(lambda: make_pipeline(char_vec(), lr()), X, cv, g)
        runs[f"lr_word_{feat_name}_{split_name}"] = cv_predict(lambda: make_pipeline(word_vec(), lr()), X, cv, g)
        runs[f"mlp_char_{feat_name}_{split_name}"] = cv_predict(lambda: make_pipeline(char_vec(), mlp()), X, cv, g)
        runs[f"nn_char_{feat_name}_{split_name}"] = nn_predict(X, cv, g)

sep_mask = df["date"].str.startswith("2026-09").to_numpy()
table = pd.DataFrame([{"run": k, "all 211 charges": (v[0] == y).mean(), "68 September charges": (v[0][sep_mask] == y[sep_mask]).mean()}
                      for k, v in runs.items()]).set_index("run")
table.round(3)

**Memorizing check.** Fit on all 211 charges and score on the same charges. A big gap between this and the "new stores" score means the model memorized store names.

In [ ]:
for name, make in [("classifier", lambda: make_pipeline(char_vec(), lr())), ("neural net", lambda: make_pipeline(char_vec(), mlp()))]:
    m = make().fit(X_clean, y)
    print(f"{name}: {(m.predict(X_clean) == y).mean():.0%} on its own training charges")

## 4. Best possible score from the text alone

If a method always gives each store its most common category, how often is it right? Charges where the same store text meant different purchases can't be fixed from the text.

In [ ]:
maj = df.groupby("merchant")["category"].agg(lambda s: s.value_counts().index[0])
mixed = df.groupby("merchant")["category"].nunique()
mixed_merchants = mixed[mixed > 1].index.tolist()
print("Stores that sell more than one kind of thing:", mixed_merchants)
print("Charges at those stores:", int(df["merchant"].isin(mixed_merchants).sum()), "of", len(df))
print(f"Best possible from text, all charges: {(df['merchant'].map(maj) == df['category']).mean():.1%}")
print(f"Best possible from text, September: {(df.loc[sep_mask, 'merchant'].map(maj) == df.loc[sep_mask, 'category']).mean():.1%}")

## 5. Classifier confidence on new stores

Our first plan: let the classifier label a charge when it's confident, and ask the student otherwise. This shows how often it was right at each confidence level, on stores it had never seen.

In [ ]:
p_lr_unseen, conf_lr_unseen = runs["lr_char_clean_unseen"]
rows = []
for t in [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf_lr_unseen >= t
    rows.append({"confidence at least": t, "charges": int(keep.sum()),
                 "right": int((p_lr_unseen[keep] == y[keep]).sum())})
print(pd.DataFrame(rows).to_string(index=False))

wrong = pd.DataFrame({"store": X_clean, "classifier said": p_lr_unseen, "confidence": conf_lr_unseen.round(2), "actually": y})
wrong[wrong["classifier said"] != wrong["actually"]].drop_duplicates("store").sort_values("confidence", ascending=False).head(10)

## 6. Similarity scores for look-alike charges

Cosine similarity between pairs of cleaned charges. Some pairs look almost the same as text but belong in different categories.

In [ ]:
v_all = char_vec().fit(X_clean)
pairs = [("UBER * EATS HELP.UBER.COM CA", "UBER * TRIP HELP.UBER.COM CA"),
         ("DOORDASH * DASHPASS SAN FRANCISCO CA", "DOORDASH * THAI THANI SAN FRANCISCO CA"),
         ("AMAZON PRIME * AMZN.COM/BILL WA", "AMAZON.COM * AMZN.COM/BILL WA"),
         ("CITY OF TROY PARKING VIOLATION TROY NY", "CITY OF TROY PARKING TROY NY"),
         ("SQ * KOPI KOPI TROY NY", "STARBUCKS STORE TROY NY")]
for a, b in pairs:
    cos = (v_all.transform([a]) @ v_all.transform([b]).T).toarray()[0, 0]
    print(f"{cos:.2f}  {a}  vs  {b}")

## 7. Recurring-bill check

A plain rule, no AI: a charge is a monthly bill if the same cleaned store text shows up about every 30 days (25 to 35 days apart) with about the same amount (within 5%). We compare against the bills we marked by hand.

In [ ]:
df["date_dt"] = pd.to_datetime(df["date"])
out_rows = df[df["amount"] < 0].copy()


def detect(rows, tol):
    found = set()
    for key, g in rows.groupby("clean"):
        g = g.sort_values("date_dt")
        if len(g) < 2:
            continue
        amts = g["amount"].abs().to_numpy()
        gaps = g["date_dt"].diff().dt.days.dropna().to_numpy()
        ok = 0
        for i in range(1, len(g)):
            same_amt = abs(amts[i] - amts[i - 1]) <= tol * max(amts[i], amts[i - 1])
            if 25 <= gaps[i - 1] <= 35 and same_amt:
                ok += 1
        if ok >= 1 and ok >= (len(g) - 1) * 0.5:
            found.add(key)
    return found


truth = set(out_rows.loc[out_rows["recurring_bill"] == 1, "clean"])
found = detect(out_rows, 0.05)
print(f"Found {len(found & truth)} of {len(truth)} bills")
print("Missed:", sorted(truth - found))
print("False alarms:", sorted(found - truth))
print()
print(df.loc[df["clean"].str.contains("NATIONAL GRID"), ["date", "amount"]].to_string(index=False))

## 8. September totals (the answer key for the LLM test)

Spending means every category except Income and Transfers. These are the numbers the LLM's totals were checked against.

In [ ]:
spend_cats = [c for c in sorted(df["category"].unique()) if c not in ("Income", "Transfers")]
sep = df[sep_mask]
spend = sep[sep["category"].isin(spend_cats)]
print(len(sep), "September charges")
print(f"Total spending: ${-spend['amount'].sum():,.2f}")
print((-spend.groupby("category")["amount"].sum()).round(2).sort_values(ascending=False).to_string())

## 9. Scoring the LLM replies (optional)

The LLM runs were done by pasting the September statement into a chat with no tools: Claude Haiku 5.5 three times (`B1`–`B3`) and Claude Opus 5.5 once (`B4`). The prompts and replies are saved in the `llm` folder. This cell checks each reply against our labels and our code's total. It skips itself if the folder isn't there.

In [ ]:
if os.path.isdir("llm"):
    truth_sep = pd.read_csv("llm/september_truth.csv").set_index("line")
    SPEND = ["Groceries", "Dining", "Transport", "Subscriptions", "Rent & Utilities", "Shopping", "Health",
             "Entertainment", "Fees"]
    labels = {}
    for name in ["B1_small", "B2_small", "B3_small", "B4_large"]:
        txt = html.unescape(open(f"llm/{name}.txt").read())
        cats, sure = {}, {}
        for m in re.finditer(r"^\s*(\d+)\s*\|\s*([^|\n]+?)\s*\|\s*(sure|unsure)\b", txt, re.M | re.I):
            cats[int(m.group(1))], sure[int(m.group(1))] = m.group(2).strip(), m.group(3).lower()
        labels[name] = cats
        t = truth_sep.copy()
        t["llm"], t["sure"] = pd.Series(cats), pd.Series(sure)
        t["ok"] = t["llm"] == t["category"]
        m = re.search(r"TOTAL SPENDING[^$\n]*\$([\d,]+\.\d\d)", txt)
        stated = m.group(1) if m else "?"
        s, u = t[t["sure"] == "sure"], t[t["sure"] == "unsure"]
        print(f"{name}: {int(t['ok'].sum())} of {len(t)} match our labels | sure: {int(s['ok'].sum())} of {len(s)} right "
              f"| unsure: {int(u['ok'].sum())} of {len(u)} right | stated total ${stated}")
    base = labels["B1_small"]
    for name in ["B2_small", "B3_small", "B4_large"]:
        diff = sorted(l for l in base if labels[name].get(l) != base.get(l))
        print(f"{name} labels that differ from B1: {diff}")
else:
    print("No llm folder here, skipping.")

## 10. The numbers on the slides

Everything on slides 4, 5 and 7 comes from the cells above, using the cleaned text and the 68 September charges.

In [ ]:
def sep_acc(key):
    return f"{(runs[key][0][sep_mask] == y[sep_mask]).mean():.0%}"


print("Keyword rules (Sept):", f"{(df.loc[sep_mask, 'rule_pred'] == df.loc[sep_mask, 'category']).mean():.0%}")
for label, key in [("Classifier", "lr_char_clean"), ("Neural net", "mlp_char_clean"), ("Similarity search", "nn_char_clean")]:
    print(f"{label}: {sep_acc(key + '_seen')} seen stores, {sep_acc(key + '_unseen')} new stores")
keep = conf_lr_unseen >= 0.7
print(f"Classifier, new stores, confidence >= 0.70: {int((p_lr_unseen[keep] == y[keep]).sum())} of {int(keep.sum())} right")